In [76]:
# import dependencies
import requests
import pandas as pd
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from alpaca.data.enums import DataFeed
from datetime import datetime

# get data - using stocks as an example but can be any time series with categories (e.g. symbols)
url = "https://en.wikipedia.org/wiki/S%26P_100"
headers = {"User-Agent": "myproject/0.1 (robert.c.dillingham@gmail.com)"}

resp = requests.get(url, headers=headers, timeout=30)
resp.raise_for_status()

tables = pd.read_html(StringIO(resp.text))
sp100 = next(t for t in tables if "Symbol" in t.columns)
symbols = sp100["Symbol"].tolist()

# limit data for testing
symbols = symbols[:10]

# get data
client = StockHistoricalDataClient("KEY", "SECRET")
req = StockBarsRequest(
    symbol_or_symbols=symbols[:10],  # batch in chunks of ~100-200
    timeframe=TimeFrame.Hour,
    start=datetime(2026, 1, 1),
    feed=DataFeed.IEX,
)
df = client.get_stock_bars(req).df

# assign further groupings & weights
# the 'keys' dict tells the function how the data should be summarized and the weights to apply to each for the weighted average matrix
keys = {'hourly': 4, 'daily': 3, 'weekly': 2, 'monthly': 1}
keys = {j[0]: j[1]/sum([i for i in keys.values()]) for j in keys.items()}

# keys need to map to a specific 'freq' value for the .groupby() function
operations = {
    'hourly': "h"
    , 'daily': "D"
    , 'weekly': "W"
    , 'monthly': "ME"
}

# because we're using US stocks, convert into NYC time zone
data = df["high"].tz_convert("America/New_York", level="timestamp")
dt = {}

# standardize each dataset, using the "High" for each stock
for tick in symbols[:10]:
    dt[tick] = {'high': data.loc[tick], 'stand': 1 + ((data.loc[tick] - max(data.loc[tick])) / max(data.loc[tick]))}

# make a dictionary entry for each summary level under each symbol. Interpolate any NaN values
for k in keys:

    for tick in symbols:
        dt[tick][k] = dt[tick]['stand'].groupby(
            pd.Grouper(level="timestamp", freq=operations[k])
        ).mean()
        dt[tick][k] = dt[tick][k].interpolate()

# make a dictionary entry for each summary level containing a matrix of r^2 values between each symbol
corr_dict = {}
for k in keys:
    m = pd.DataFrame(0.0, index=symbols, columns=symbols)

    for s1 in symbols:

        for s2 in symbols:
            m.loc[s1, s2] = dt[s1][k].corr(dt[s2][k]) ** 2

    corr_dict[k] = m

# create a final matrix of weighted average r^2 values to be used as clustering & predictive values
corr_dict['result'] = sum([m[0][1]*m[1][1] for m in zip(corr_dict.items(), keys.items())])
        

In [2]:
import requests
import pandas as pd
from io import StringIO

url = "https://en.wikipedia.org/wiki/S%26P_100"
headers = {"User-Agent": "myproject/0.1 (robert.c.dillingham@gmail.com)"}

resp = requests.get(url, headers=headers, timeout=30)
resp.raise_for_status()

tables = pd.read_html(StringIO(resp.text))
sp100 = next(t for t in tables if "Symbol" in t.columns)
symbols = sp100["Symbol"].tolist()



In [7]:
client = StockHistoricalDataClient("KEY", "SECRET")
req = StockBarsRequest(
    symbol_or_symbols=symbols[:10],  # batch in chunks of ~100-200
    timeframe=TimeFrame.Hour,
    start=datetime(2026, 1, 1),
    feed=DataFeed.IEX,
)
df = client.get_stock_bars(req).df

In [58]:
symbols

['AAPL', 'ABBV', 'ABT', 'ACN', 'ADBE', 'AMAT', 'AMD', 'AMGN', 'AMT', 'AMZN']

In [55]:
dt[tick][k] = dt[tick][k].interpolate()

In [69]:
keys = {'hourly': 4, 'daily': 3, 'weekly': 2, 'monthly': 1}
keys = {j[0]: j[1]/sum([i for i in keys.values()]) for j in keys.items()}

operations = {
    'hourly': "h"
    , 'daily': "D"
    , 'weekly': "W"
    , 'monthly': "ME"
}

data = df["high"].tz_convert("America/New_York", level="timestamp")
dt = {}

for tick in symbols[:10]:
    dt[tick] = {'high': data.loc[tick], 'stand': 1 + ((data.loc[tick] - max(data.loc[tick])) / max(data.loc[tick]))}

for k in keys:

    for tick in symbols:
        dt[tick][k] = dt[tick]['stand'].groupby(
            pd.Grouper(level="timestamp", freq=operations[k])
        ).mean()
        dt[tick][k] = dt[tick][k].interpolate()

corr_dict = {}
for k in keys:
    m = pd.DataFrame(0.0, index=symbols, columns=symbols)

    for s1 in symbols:

        for s2 in symbols:
            m.loc[s1, s2] = dt[s1][k].corr(dt[s2][k]) ** 2

    corr_dict[k] = m

corr_dict['result'] = sum([m[0][1]*m[1][1] for m in zip(corr_dict.items(), keys.items())])
        

,AAPL,ABBV,ABT,ACN,ADBE,AMAT,AMD,AMGN,AMT,AMZN
AAPL,1.000000,0.717813,0.040212,0.030806,0.109710,0.217701,0.796961,0.502939,0.002140,0.386160
ABBV,0.717813,1.000000,0.049362,0.010749,0.020596,0.074995,0.439351,0.792743,0.001322,0.110188
ABT,0.040212,0.049362,1.000000,0.440757,0.502671,0.295586,0.209994,0.097610,0.004359,0.125189
ACN,0.030806,0.010749,0.440757,1.000000,0.504179,0.661361,0.193741,0.031614,0.159495,0.027971
ADBE,0.109710,0.020596,0.502671,0.504179,1.000000,0.451296,0.254870,0.007064,0.009343,0.006875
AMAT,0.217701,0.074995,0.295586,0.661361,0.451296,1.000000,0.564418,0.010241,0.166499,0.144499
AMD,0.796961,0.439351,0.209994,0.193741,0.254870,0.564418,1.000000,0.225960,0.045424,0.473921
AMGN,0.502939,0.792743,0.097610,0.031614,0.007064,0.010241,0.225960,1.000000,0.028473,0.073541
AMT,0.002140,0.001322,0.004359,0.159495,0.009343,0.166499,0.045424,0.028473,1.000000,0.011802
AMZN,0.386160,0.110188,0.125189,0.027971,0.006875,0.144499,0.473921,0.073541,0.011802,1.000000
